# 12 — Advanced Tuning (XGBoost & CatBoost)
This notebook provides Optuna tuning cells for our remaining two boosters. Tuning these will ensure our final ensemble is composed of five high-performing "experts."

In [3]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/santander-customer-satisfaction/'
PICKLE_DIR = DRIVE_PATH + 'pickles/'

import pandas as pd
import numpy as np
!pip install optuna
!pip install catboost
import optuna
import xgboost as xgb
from catboost import CatBoostClassifier
import pickle
from sklearn.metrics import roc_auc_score

train = pd.read_pickle(f'{PICKLE_DIR}train_advanced.pkl')
X = train.drop(columns=['TARGET', 'ID'], errors='ignore')
y = train['TARGET']

with open(f'{PICKLE_DIR}cv_fold_indices.pkl', 'rb') as f:
    cv_folds = pickle.load(f)

print('Data loaded. Ready for tuning.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.9 MB/s eta 0:00:00
Data loaded. Ready for tuning.


## 1. XGBoost Tuning (GPU)

In [7]:
from xgboost.callback import EarlyStopping
import xgboost as xgb

def xgb_objective(trial):
    param = {
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'tree_method': 'hist', # Changed from 'gpu_hist' to 'hist'
        'lambda': trial.suggest_float('lambda', 1e-8, 10.0, log=True),
        'alpha': trial.suggest_float('alpha', 1e-8, 10.0, log=True),
        'subsample': trial.suggest_float('subsample', 0.4, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 100),
        'eta': trial.suggest_float('eta', 0.005, 0.1, log=True),
        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
        'verbosity': 0 # Added verbosity to param for xgb.train
    }

    fold_aucs = []
    for fold_dict in cv_folds:
        X_train_fold, y_train_fold = X.iloc[fold_dict['train_idx']], y.iloc[fold_dict['train_idx']]
        X_val_fold, y_val_fold = X.iloc[fold_dict['val_idx']], y.iloc[fold_dict['val_idx']]

        # Convert data to DMatrix for xgb.train
        dtrain = xgb.DMatrix(X_train_fold, label=y_train_fold)
        dval = xgb.DMatrix(X_val_fold, label=y_val_fold)

        # Define the early stopping callback
        early_stopping_callback = EarlyStopping(
            rounds=100, # Number of rounds with no improvement after which training stops.
            save_best=True, # Whether to save the best model found
            maximize=True # Whether to maximize (True) or minimize (False) the evaluation metric
        )

        # Use xgb.train instead of XGBClassifier.fit()
        model_bst = xgb.train(
            param,
            dtrain,
            num_boost_round=2000, # Max number of boosting rounds
            evals=[(dval, 'validation')],
            callbacks=[early_stopping_callback],
            verbose_eval=False
        )

        # Get predictions from the best iteration of the trained model
        preds = model_bst.predict(dval, iteration_range=(0, model_bst.best_iteration + 1))
        fold_aucs.append(roc_auc_score(y_val_fold, preds))

    return np.mean(fold_aucs)

xgb_study = optuna.create_study(direction='maximize')
xgb_study.optimize(xgb_objective, n_trials=30)

print("Best XGB AUC:", xgb_study.best_value)
print("Best XGB Params:", xgb_study.best_params)

[I 2026-05-02 23:35:06,365] A new study created in memory with name: no-name-c92f07af-1852-4f2b-8350-b3214da9b494
[I 2026-05-02 23:36:35,287] Trial 0 finished with value: 0.83750524383547 and parameters: {'lambda': 0.001872970949044569, 'alpha': 0.005966612537481622, 'subsample': 0.6610091787038244, 'colsample_bytree': 0.4847246889419091, 'max_depth': 10, 'min_child_weight': 90, 'eta': 0.01651907814273129, 'gamma': 2.8417529301395984e-06}. Best is trial 0 with value: 0.83750524383547.
[I 2026-05-02 23:37:58,133] Trial 1 finished with value: 0.8398482456073945 and parameters: {'lambda': 1.743049516637411e-06, 'alpha': 0.001023793293591234, 'subsample': 0.5728173513421406, 'colsample_bytree': 0.46037386578824546, 'max_depth': 9, 'min_child_weight': 12, 'eta': 0.00773322669171119, 'gamma': 0.17515595114225532}. Best is trial 1 with value: 0.8398482456073945.
[I 2026-05-02 23:39:03,481] Trial 2 finished with value: 0.8387185058581798 and parameters: {'lambda': 0.040482481022253526, 'alpha'

Best XGB AUC: 0.8408815622380736
Best XGB Params: {'lambda': 0.40525764696520994, 'alpha': 0.004209910021977267, 'subsample': 0.8492306700124232, 'colsample_bytree': 0.4697530115943648, 'max_depth': 4, 'min_child_weight': 2, 'eta': 0.01704610324611452, 'gamma': 0.00031128353436984593}


## 2. CatBoost Tuning (GPU)

In [9]:
def cat_objective(trial):
    param = {
        'iterations': 1500,
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-8, 10.0, log=True),
        'random_strength': trial.suggest_float('random_strength', 1e-8, 10.0, log=True),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 10.0),
        'od_type': 'Iter',
        'od_wait': 50,
        'task_type': 'GPU',
        'verbose': False,
        'eval_metric': 'AUC',
        'random_seed': 42
    }

    fold_aucs = []
    for fold_dict in cv_folds:
        X_train, y_train = X.iloc[fold_dict['train_idx']], y.iloc[fold_dict['train_idx']]
        X_val, y_val = X.iloc[fold_dict['val_idx']], y.iloc[fold_dict['val_idx']]

        model = CatBoostClassifier(**param)
        model.fit(X_train, y_train, eval_set=(X_val, y_val), use_best_model=True)

        # Removed erroneous line: preds = model.get_test_set_scale_and_bias_for_intermediate_prediction_logic_check_only_predict_proba(X_val)[:, 1]
        # Correction for CatBoost predict logic in Optuna loops:
        preds = model.predict_proba(X_val)[:, 1]

        fold_aucs.append(roc_auc_score(y_val, preds))

    return np.mean(fold_aucs)

cat_study = optuna.create_study(direction='maximize')
cat_study.optimize(cat_objective, n_trials=30)

print("Best CatBoost AUC:", cat_study.best_value)
print("Best CatBoost Params:", cat_study.best_params)

[I 2026-05-03 00:23:06,888] A new study created in memory with name: no-name-a6ba8a54-9122-43ce-96eb-313298f650ba
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
[I 2026-05-03 00:23:46,136] Trial 0 finished with value: 0.83757061628209 and parameters: {'learning_rate': 0.0708102541864767, 'depth': 6, 'l2_leaf_reg': 2.427648416060749, 'random_strength': 0.023089148334018206, 'bagging_temperature': 7.111759846143468}. Best is trial 0 with value: 0.83757061628209.
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period 

Best CatBoost AUC: 0.8391964982943734
Best CatBoost Params: {'learning_rate': 0.05113275824082385, 'depth': 6, 'l2_leaf_reg': 7.658112640634797, 'random_strength': 8.934300157134556e-06, 'bagging_temperature': 5.277232910748792}
